In [ ]:
# This notebook is derived from Qiskit and includes modifications by qBraid.
#
# (C) Copyright IBM 2020.
# (C) Copyright qBraid 2024.
#
# This code is licensed under the Apache License, Version 2.0. You may
# obtain a copy of this license in the LICENSE.txt file in the root directory
# of this source tree or at http://www.apache.org/licenses/LICENSE-2.0.
#
# Any modifications or derivative works of this code must retain this
# copyright notice, and modified files need to carry a notice indicating
# that they have been altered from the originals.

# Grover's Algorithm with qBraid

This notebook demonstrates Grover's quantum search algorithm using:

1. **qbraid-algorithms**: The QTRAN framework to generate Grover's algorithm as OpenQASM 3.0
2. **Qiskit**: To visualize the generated circuits
3. **qBraid Runtime**: To run on a local Aer simulator or a real IBM Quantum backend

Grover's algorithm finds a marked item in an unsorted database of N items in approximately sqrt(N) steps, providing a quadratic speedup over classical search. We run Grover's for varying iteration counts (0 to 3) to observe how the success probability changes.

In [ ]:
%matplotlib inline

# Toggle this flag to switch between local simulator and real IBM backend
USE_SIMULATOR = True

## Part 1: Building Grover's Algorithm with qbraid-algorithms

The `qbraid-algorithms` package provides the QTRAN framework for constructing quantum algorithms as OpenQASM 3.0. We define a custom oracle as a `GateLibrary` and use the `QasmBuilder` to compose the full Grover circuit — initialization, oracle, and diffuser — directly as executable QASM.

In [ ]:
from qbraid_algorithms.qtran import QasmBuilder, std_gates, GateLibrary


class GroverOracle(GateLibrary):
    """Oracle that marks the state |101> (decimal 5) for a 3-qubit search."""

    name = "oracle"

    def apply(self, qubits):
        # Phase-flip |101>: X on q[1] to flip it, then CCZ, then X to undo
        self.call_gate("x", qubits[1])
        self.controlled_op("z", (qubits[0], [qubits[1], qubits[2]]), n=2)
        self.call_gate("x", qubits[1])

    def unapply(self, qubits):
        self.apply(qubits)  # This oracle is self-inverse

In [ ]:
n = 3


def build_grover_qasm(n_qubits, depth):
    """Use QTRAN to build a Grover search circuit as executable QASM 3.0."""
    builder = QasmBuilder(qubits=n_qubits, clbits=n_qubits)
    std_lib = builder.import_library(std_gates)
    oracle_lib = builder.import_library(GroverOracle)
    std_lib.call_space = " {}"
    oracle_lib.call_space = " {}"

    qubits = [f"qb[{i}]" for i in range(n_qubits)]

    # Initialization: uniform superposition
    std_lib.h("qb")

    # Grover iterations: oracle + diffuser
    for _ in range(depth):
        oracle_lib.apply(qubits)
        std_lib.h("qb")
        std_lib.x("qb")
        std_lib.controlled_op("z", (qubits[-1], qubits[:-1]), n=n_qubits - 1)
        std_lib.x("qb")
        std_lib.h("qb")

    builder.program_append("cb = measure qb;")
    return builder.build()


# Show the generated QASM for 1 iteration
qasm_example = build_grover_qasm(n, depth=1)
print("Generated OpenQASM 3.0 (1 Grover iteration):")
print("=" * 50)
print(qasm_example)

## Part 2: Loading and Visualizing the Circuits

We load the QTRAN-generated QASM into Qiskit with `qasm3.loads()` to visualize the circuits and prepare them for execution.

In [ ]:
from qiskit.qasm3 import loads as qasm3_loads

# Generate QASM programs for 0-3 Grover iterations and load into Qiskit
n_steps = 3
circuits = []

for depth in range(n_steps + 1):
    qasm = build_grover_qasm(n, depth)
    circuits.append(qasm3_loads(qasm))

In [ ]:
# Draw the circuit with 2 Grover iterations (near-optimal for N=8)
print(f"Circuit with 2 Grover iterations ({circuits[2].num_qubits} qubits):")
circuits[2].draw()

## Part 3: Running the Circuits

When `USE_SIMULATOR = True`, we run locally on `AerSimulator`. Otherwise, we use `QiskitRuntimeProvider` from the qBraid SDK to submit to a real IBM Quantum backend.

In [ ]:
if USE_SIMULATOR:
    from qiskit_aer import AerSimulator

    backend = AerSimulator()
    print(f"Using local simulator: {backend.name}")
else:
    import os

    from qbraid.runtime import QiskitRuntimeProvider

    token = os.getenv("QISKIT_IBM_TOKEN") or ""
    instance = os.getenv("QISKIT_IBM_INSTANCE") or ""

    provider = QiskitRuntimeProvider(token=token, instance=instance)
    print("Available devices:")
    print(provider.get_devices())

In [ ]:
from qiskit import transpile

shots = 1024
batch_counts = []

if USE_SIMULATOR:
    transpiled = transpile(circuits, backend)
    for i, circ in enumerate(transpiled):
        job = backend.run(circ, shots=shots)
        counts = job.result().get_counts()
        batch_counts.append(counts)
        print(f"Circuit {i} ({i} iterations): {counts}")
else:
    device = provider.get_device("ibm_kingston")  # replace with your preferred backend
    for i, circ in enumerate(circuits):
        qbraid_job = device.run(circ, shots=shots)
        result = qbraid_job.result()
        counts = result.data.get_counts()
        batch_counts.append(counts)
        print(f"Circuit {i} ({i} iterations): {counts}")

## Part 4: Visualize Results

In [ ]:
from qbraid.visualization import plot_histogram

for i, counts in enumerate(batch_counts):
    print(f"\n{'='*40}")
    print(f"Grover iterations: {i}")
    print(f"{'='*40}")
    plot_histogram(counts)

### Interpreting the Results

For a 3-qubit search space (N = 8) with a single marked state, the optimal number of Grover iterations is approximately &pi;/4 &middot; &radic;N &asymp; **2.2**. This is reflected in the results:

- **0 iterations**: All 8 states appear with roughly equal probability (~12.5% each) — this is just the uniform superposition with no amplification.
- **1 iteration**: The marked state `|101>` already dominates (~78%), demonstrating rapid amplitude amplification.
- **2 iterations**: `|101>` reaches near-peak probability (~94%), closest to the theoretical optimum.
- **3 iterations**: The probability of `|101>` drops significantly (~35%) as the algorithm **overshoots** — Grover's search is periodic, and too many iterations rotate the state vector past the solution and back toward the diffused (uniform) state.

This periodic behavior is a key feature of Grover's algorithm: more iterations are not always better. The number of iterations must be tuned to approximately &pi;/4 &middot; &radic;N for optimal success probability.

<div class="alert alert-block alert-info">
<b>Copyright Notice:</b> 
    All rights reserved © [2026] qBraid. This notebook is part of the qBraid-Lab-Demo repository.
The qBraid-Lab-Demo is licensed under the Apache License, Version 2.0.
You may obtain a copy of the License at <https://www.apache.org/licenses/LICENSE-2.0>.
Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
</div>